# Project 1

We do a 7-class emotion classification on the ISEAR dataset. Our goal is to compare different representations and architectures with cross validation and Bayesian estimation.

## Imports

In [36]:
# Imports
import sys
from pathlib import Path

root = Path.cwd().parent
sys.path.insert(0, str(root))

import part1
import smlp

import datasets
import numpy as np
import re

from tokenizers import Tokenizer
from tabulate import tabulate
from collections import OrderedDict

from itertools import combinations
import pandas as pd

import torch
import torch.nn as nn
from scipy.sparse import csr_matrix
from torch.utils.data import Dataset
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm
import scipy
import scipy.sparse
import json

## Dataset

In [2]:

dataset=datasets.load_dataset("gsri-18/ISEAR-dataset-complete") # we use ISEAR for the projects

In [3]:
# Small dataset statistic cell

print("Names of the splits: ", dataset.keys(), "\n"*2)
print("Number of rows: ", dataset.num_rows, "\n"*2)
print("A single Dataset: ", dataset["train"]), "\n"*2
print("Names of the splits: ", dataset["train"].features, "\n"*2)
print("Column names: ", dataset["train"].column_names, "\n"*2)
print("Rows: ", dataset["train"].num_rows, "\n"*2)
print("One example as dict: ", dataset["train"][0], "\n"*2)
print("First 5 rows as a dict of lists: ", dataset["train"][:5], "\n"*2)
print("Whole column as a list: ", dataset["train"]["content"], "\n"*2)
print("Unique emotions: ", dataset["train"].unique("emotion"), "\n"*2)


Names of the splits:  dict_keys(['train']) 


Number of rows:  {'train': 7516} 


A single Dataset:  Dataset({
    features: ['emotion', 'content', 'Unnamed: 2'],
    num_rows: 7516
})
Names of the splits:  {'emotion': Value('string'), 'content': Value('string'), 'Unnamed: 2': Value('string')} 


Column names:  ['emotion', 'content', 'Unnamed: 2'] 


Rows:  7516 


One example as dict:  {'emotion': 'joy', 'content': 'On days when I feel close to my partner and other friends.   \nWhen I feel at peace with myself and also experience a close  \ncontact with people whom I regard greatly.', 'Unnamed: 2': None} 


First 5 rows as a dict of lists:  {'emotion': ['joy', 'fear', 'anger', 'sadness', 'disgust'], 'content': ['On days when I feel close to my partner and other friends.   \nWhen I feel at peace with myself and also experience a close  \ncontact with people whom I regard greatly.', 'Every time I imagine that someone I love or I could contact a  \nserious illness, even death.', 'When I 

In [4]:
# normalize and preprocess text

def normalise_text(text: str) -> str:
    # any whitespace into single blank space
    return re.sub(r"\s+", " ", text).strip()

# when inspecting the dataset there were 2 empty columns, remove them (it is also visible on HF)
if "Unnamed: 2" in dataset["train"].column_names:
    dataset = dataset.remove_columns("Unnamed: 2")

print("Before preprocessing: ", dataset["train"][0]["content"], "\n")
dataset = dataset.map(lambda ex: {"content": normalise_text(ex["content"])})

print("After preprocessing: ", dataset["train"][0]["content"])
print("Docs still containing a newline:", sum("\n" in t for t in dataset["train"]["content"]))
print("Docs with double spaces:", sum("  " in t for t in dataset["train"]["content"]))


Before preprocessing:  On days when I feel close to my partner and other friends.   
When I feel at peace with myself and also experience a close  
contact with people whom I regard greatly. 



Map:   0%|          | 0/7516 [00:00<?, ? examples/s]

After preprocessing:  On days when I feel close to my partner and other friends. When I feel at peace with myself and also experience a close contact with people whom I regard greatly.
Docs still containing a newline: 0
Docs with double spaces: 0


In [5]:
agnews_outcome_space = part1.clslib.NamedClasses(['anger', 'disgust', 'fear', 'guilt', 'joy', 'sadness', 'shame'])
label_mapping = {name: i for i,name in enumerate(agnews_outcome_space)}
dataset = dataset.map(lambda example: {'emotion': label_mapping[example['emotion']]})


Map:   0%|          | 0/7516 [00:00<?, ? examples/s]

In [6]:

corpus = dataset["train"]["content"]
len(corpus)
vocab_size = 1000
min_frequency = 2

results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True)

tokenizer = part1.experiment.train_tokenizer(corpus, vocab_size, min_frequency)
tokenizer.save(f"{results_folder}/BPE_tokenizer_vs_{vocab_size}_mf_{min_frequency}.json")

#corpus_testing = dataset["test"]["content"]


In [15]:
# my_seeds = [0, 3, 5]
my_seeds = [0] # to add less seeds which saves comp time

num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

for seed in my_seeds:
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(dataset["train"]["content"], dataset["train"]["emotion"])),
            k=num_folds,
            seed=seed
        )
    )

In [8]:
my_classifiers = OrderedDict()

In [28]:
# shared settings, so every system is trained with the same hyperparameters
loglin_args = {'max_iter': 100}
smlp_args = {
    'hidden_dim': 128,
    'dropout': 0.1,
    'batch_size': 64,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'num_epochs': 30,
}
two_layer_mlp_args = {
    'hidden_dim1': 128,
    'hidden_dim2': 64,
    'dropout': 0.1,
    'batch_size': 64,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'num_epochs': 30,
}


In [10]:
class Encoder:

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")

In [11]:
class TokenIndicator(Encoder):

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(self.tokenizer.encode(x).ids))

        # A 0 vector with the same length as the vocabulary except in positions corresponding
        # to tokens appearing in the document
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector

In [12]:
token_indicator_encoder = TokenIndicator(tokenizer)
doc_repr = token_indicator_encoder(corpus[0])

In [13]:
class BagOfWords(Encoder):
    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
    def __len__(self) -> int:
            """Dimensionality of the representation space"""
            return len(tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(self.tokenizer.encode(x).ids)

        # A 0 vector with the same length as the vocabulary except in positions corresponding
        # to tokens appearing in the document
        vector = np.zeros(len(self))
        for id in unique_token_ids:
            vector[id] += 1
        return vector

bow_encoder = BagOfWords(tokenizer)

# TF IDF

Implementation is based on https://medium.com/bitgrit-data-science-publication/tf-idf-from-scratch-in-python-ea587d003e9e

In [14]:
def tf(word_counts: dict, document: list) -> dict:
    """Calculate term frequency of each word in a document."""
    tf_dict = {}
    corpus_count = max(len(document), 1)   # guard against empty documents

    for word, count in word_counts.items():
        tf_dict[word] = count / float(corpus_count)

    return tf_dict


def idf(documents: list, vocab) -> dict:
    """For each term, the (smoothed) inverse of the number of documents it appears in."""
    N = len(documents)
    idf_dict = dict.fromkeys(vocab, 0)

    for doc in documents:
        for word in set(doc):          # count each term once per document
            idf_dict[word] += 1

    for word, df in idf_dict.items():
        idf_dict[word] = np.log10((N + 1.0) / (df + 1.0))

    return idf_dict


def tfidf(doc_elements: dict, idfs: dict) -> dict:
    """TF * IDF per word given a single document."""
    tfidf_dict = {}

    for word, val in doc_elements.items():
        tfidf_dict[word] = val * idfs[word]

    return tfidf_dict


class TfIdf(Encoder):

    def __init__(self, tokenizer: Tokenizer, fit_corpus: list):
        self.tokenizer = tokenizer
        self.vocab = range(tokenizer.get_vocab_size())   # token ids are our "words"
        self.idfs = idf([tokenizer.encode(text).ids for text in fit_corpus], self.vocab)

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> 'np.array':
        document = self.tokenizer.encode(x).ids

        word_counts = dict.fromkeys(self.vocab, 0)
        for word in document:
            word_counts[word] += 1

        scores = tfidf(tf(word_counts, document), self.idfs)
        vector = np.array([scores[word] for word in self.vocab])

        norm = np.linalg.norm(vector)                    # L2-normalise (see note above)
        return vector / norm if norm > 0 else vector


tfidf_encoder = TfIdf(tokenizer, corpus)

vec = tfidf_encoder(corpus[0])
print(vec.shape, "non-zero entries:", np.count_nonzero(vec))
top = np.argsort(vec)[::-1][:5]
print([(tokenizer.id_to_token(int(i)), round(float(vec[i]), 3)) for i in top])


(1000,) non-zero entries: 34
[('▁close', 0.321), ('▁feel', 0.307), ('▁pe', 0.263), ('ace', 0.216), ('ner', 0.211)]


# Two-Layer MLP

In [23]:
class TwoLayerSparseMLP(nn.Module):

    def __init__(self, input_dim, hidden_dim1, hidden_dim2, num_classes=7, dropout=0.2):
        super().__init__()

        self.weight = nn.Parameter(torch.empty(input_dim, hidden_dim1))
        self.bias = nn.Parameter(torch.zeros(hidden_dim1))

        # this is the second hidden layer
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)

        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        self.output = nn.Sequential(
            nn.Linear(hidden_dim2, num_classes), 
            nn.LogSoftmax(dim=-1)
        )

        nn.init.xavier_uniform_(self.weight)
        nn.init.xavier_uniform_(self.fc2.weight)

    def forward(self, x):        
        x = torch.sparse.mm(x, self.weight)
        x = x + self.bias
        x = self.activation(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.activation(x)
        x = self.dropout(x)

        return self.output(x)

In [40]:
def train_twolayer_nn_model(training_set, validation_set, num_classes, encoder, save_to,   
                             hidden_dim1=128, hidden_dim2=64, dropout=0.1, batch_size=64, 
                             learning_rate=1e-3, weight_decay=1e-4, num_epochs=20, device=torch.device("cpu")):
    if isinstance(device, str):
        device = torch.device(device)

    training_vectors = [encoder(text) for text, label in training_set]
    training_labels = [int(label) for text, label in training_set]
    validation_vectors = [encoder(text) for text, label in validation_set]
    validation_labels = [int(label) for text, label in validation_set]

    X_train_sparse = csr_matrix(training_vectors)
    X_val_sparse = csr_matrix(validation_vectors)

    train_dataset = smlp.SparseDataset(X_train_sparse, training_labels)
    val_dataset = smlp.SparseDataset(X_val_sparse, validation_labels)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=smlp.sparse_collate
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=smlp.sparse_collate
    )

    with open(f"{save_to}.json", "w", encoding="utf-8") as f:
        args = {
            'input_dim': len(encoder),
            'hidden_dim1': hidden_dim1,
            'hidden_dim2': hidden_dim2,
            'num_classes': num_classes,
            'dropout': dropout,
        }
        json.dump(args, f, indent=2, ensure_ascii=False)
        
    model = TwoLayerSparseMLP(**args).to(device)
    criterion = nn.NLLLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    metrics = []
    print("Training...")
    predictions = None
    best_val_loss = float('inf')

    for epoch in tqdm(range(num_epochs)):
        train_metrics = smlp.train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics, preds = smlp.evaluate(model, val_loader, criterion, device)
        metrics.append({"train_metrics": train_metrics, "val_metrics": val_metrics})

        print(
            f"Epoch {epoch + 1:02d} | "
            f"train loss: {train_metrics['loss']:.4f} | "
            f"train acc: {train_metrics['accuracy']:.4f} | "
            f"val loss: {val_metrics['loss']:.4f} | "
            f"val acc: {val_metrics['accuracy']:.4f}"
            )
        
        if val_metrics["loss"] <= best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(model.state_dict(), f"{save_to}.pt")
            best_epoch = epoch
            predictions = preds

    return metrics, best_epoch, predictions

In [30]:
def twolayer_cross_val_exp(data_splits, outcome_space, encoder: 'Encoder', save_to="tmp", mlp_args=dict()):
    models, predictions, accuracies = OrderedDict(), OrderedDict(), OrderedDict()

    for fold in tqdm(data_splits.keys()):
        training_set = data_splits[fold]["training"]
        validation_set = data_splits[fold]["validation"]

        fold_folder = Path(f"{save_to}/fold-{fold}")
        fold_folder.mkdir(parents=True, exist_ok=True)

        # two layer mlp
        metrics, best_epoch, predictions[fold] = train_twolayer_nn_model(
            training_set, validation_set, len(outcome_space), encoder, save_to=f'{fold_folder}/model', **mlp_args
        )
        models[fold] = f'{fold_folder}/model'
        accuracies[fold] = metrics[best_epoch]['val_metrics']['accuracy']

    return {'models': models, 'predictions': predictions, 'accuracies': accuracies}

# Runs

In [16]:
%%time
my_classifiers['logistic_indicator-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_indicator-enc/seed-{seed}"
    my_classifiers['logistic_indicator-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=token_indicator_encoder,
            model_option='logistic',
            save_to=run_folder,
            loglin_args=loglin_args,
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/

CPU times: user 33.1 s, sys: 118 ms, total: 33.2 s
Wall time: 33.2 s


In [17]:
%%time
my_classifiers['logistic_bow-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_bow-enc/seed-{seed}"
    my_classifiers['logistic_bow-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=bow_encoder,
            model_option='logistic',
            save_to=run_folder,
            loglin_args=loglin_args,
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/

CPU times: user 31.8 s, sys: 60.9 ms, total: 31.9 s
Wall time: 31.9 s


In [18]:
%%time
my_classifiers['logistic_tfidf-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_tfidf-enc/seed-{seed}"
    my_classifiers['logistic_tfidf-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=tfidf_encoder,
            model_option='logistic',
            save_to=run_folder,
            loglin_args=loglin_args,
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be r

CPU times: user 30.1 s, sys: 80.1 ms, total: 30.2 s
Wall time: 30.2 s


In [19]:
%%time
my_classifiers['mlp_indicator-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_indicator-enc/seed-{seed}"
    my_classifiers['mlp_indicator-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=token_indicator_encoder,
            model_option='mlp',
            save_to=run_folder,
            smlp_args=smlp_args,
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 4.970 seconds
Dataloader done! Time taken: 0.131 seconds
Training...


  0%|          | 0/30 [00:00<?, ?it/s]

/home/jaimie/NLP/nlp1-student-tutorial1-2026/part1/smlp.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:249.)
  indices = torch.tensor([x.row, x.col], dtype=torch.long)
/home/jaimie/NLP/nlp1-student-tutorial1-2026/part1/smlp.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:916.)
  return torch.sparse_coo_tensor(indices, values, size=x.shape).coalesce()
/home/jaimie/miniconda3/envs/nlp/lib/python3.14/site-pa

Epoch 01 | train loss: 1.8354 | train acc: 0.3232 | val loss: 1.6673 | val acc: 0.4515
Epoch 02 | train loss: 1.3957 | train acc: 0.5584 | val loss: 1.3682 | val acc: 0.5346
Epoch 03 | train loss: 1.1067 | train acc: 0.6329 | val loss: 1.2908 | val acc: 0.5459
Epoch 04 | train loss: 0.9497 | train acc: 0.6820 | val loss: 1.2782 | val acc: 0.5572
Epoch 05 | train loss: 0.8481 | train acc: 0.7184 | val loss: 1.2900 | val acc: 0.5559
Epoch 06 | train loss: 0.7773 | train acc: 0.7450 | val loss: 1.3208 | val acc: 0.5532
Epoch 07 | train loss: 0.7091 | train acc: 0.7680 | val loss: 1.3496 | val acc: 0.5539
Epoch 08 | train loss: 0.6517 | train acc: 0.7874 | val loss: 1.3851 | val acc: 0.5532
Epoch 09 | train loss: 0.6036 | train acc: 0.8039 | val loss: 1.4318 | val acc: 0.5432
Epoch 10 | train loss: 0.5557 | train acc: 0.8230 | val loss: 1.4582 | val acc: 0.5519
Epoch 11 | train loss: 0.5114 | train acc: 0.8428 | val loss: 1.4942 | val acc: 0.5532
Epoch 12 | train loss: 0.4661 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

... (3 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8408 | train acc: 0.3198 | val loss: 1.6718 | val acc: 0.4657
Epoch 02 | train loss: 1.4033 | train acc: 0.5621 | val loss: 1.3818 | val acc: 0.5276
Epoch 03 | train loss: 1.0987 | train acc: 0.6376 | val loss: 1.3090 | val acc: 0.5416
Epoch 04 | train loss: 0.9465 | train acc: 0.6840 | val loss: 1.3027 | val acc: 0.5509
Epoch 05 | train loss: 0.8469 | train acc: 0.7184 | val loss: 1.3171 | val acc: 0.5509
Epoch 06 | train loss: 0.7646 | train acc: 0.7509 | val loss: 1.3422 | val acc: 0.5509
Epoch 07 | train loss: 0.7001 | train acc: 0.7762 | val loss: 1.3850 | val acc: 0.5469
Epoch 08 | train loss: 0.6478 | train acc: 0.7936 | val loss: 1.4193 | val acc: 0.5469
Epoch 09 | train loss: 0.5910 | train acc: 0.8112 | val loss: 1.4500 | val acc: 0.5462
Epoch 10 | train loss: 0.5434 | train acc: 0.8314 | val loss: 1.4833 | val acc: 0.5489
Epoch 11 | train loss: 0.4947 | train acc: 0.8522 | val loss: 1.5367 | val acc: 0.5383
Epoch 12 | train loss: 0.4522 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8427 | train acc: 0.3231 | val loss: 1.6590 | val acc: 0.4704
Epoch 02 | train loss: 1.4132 | train acc: 0.5580 | val loss: 1.3353 | val acc: 0.5436
Epoch 03 | train loss: 1.1159 | train acc: 0.6321 | val loss: 1.2458 | val acc: 0.5615
Epoch 04 | train loss: 0.9602 | train acc: 0.6835 | val loss: 1.2317 | val acc: 0.5675
Epoch 05 | train loss: 0.8534 | train acc: 0.7116 | val loss: 1.2411 | val acc: 0.5642
Epoch 06 | train loss: 0.7693 | train acc: 0.7480 | val loss: 1.2671 | val acc: 0.5629
Epoch 07 | train loss: 0.7091 | train acc: 0.7640 | val loss: 1.2833 | val acc: 0.5715
Epoch 08 | train loss: 0.6496 | train acc: 0.7938 | val loss: 1.3275 | val acc: 0.5689
Epoch 09 | train loss: 0.5895 | train acc: 0.8134 | val loss: 1.3609 | val acc: 0.5622
Epoch 10 | train loss: 0.5451 | train acc: 0.8329 | val loss: 1.3936 | val acc: 0.5675
Epoch 11 | train loss: 0.4929 | train acc: 0.8537 | val loss: 1.4202 | val acc: 0.5682
Epoch 12 | train loss: 0.4538 | train acc: 

In [20]:
%%time
my_classifiers['mlp_bow-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_bow-enc/seed-{seed}"
    my_classifiers['mlp_bow-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=bow_encoder,
            model_option='mlp',
            save_to=run_folder,
            smlp_args=smlp_args,
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 4.795 seconds
Dataloader done! Time taken: 0.114 seconds
Training...


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8359 | train acc: 0.3260 | val loss: 1.6728 | val acc: 0.4448
Epoch 02 | train loss: 1.4279 | train acc: 0.5437 | val loss: 1.3948 | val acc: 0.5160
Epoch 03 | train loss: 1.1324 | train acc: 0.6356 | val loss: 1.2976 | val acc: 0.5479
Epoch 04 | train loss: 0.9662 | train acc: 0.6810 | val loss: 1.2880 | val acc: 0.5485
Epoch 05 | train loss: 0.8493 | train acc: 0.7221 | val loss: 1.2988 | val acc: 0.5459
Epoch 06 | train loss: 0.7705 | train acc: 0.7513 | val loss: 1.3298 | val acc: 0.5492
Epoch 07 | train loss: 0.6940 | train acc: 0.7721 | val loss: 1.3521 | val acc: 0.5552
Epoch 08 | train loss: 0.6341 | train acc: 0.7982 | val loss: 1.3840 | val acc: 0.5565
Epoch 09 | train loss: 0.5774 | train acc: 0.8184 | val loss: 1.4265 | val acc: 0.5559
Epoch 10 | train loss: 0.5266 | train acc: 0.8370 | val loss: 1.4604 | val acc: 0.5625
Epoch 11 | train loss: 0.4770 | train acc: 0.8584 | val loss: 1.5040 | val acc: 0.5618
Epoch 12 | train loss: 0.4394 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8300 | train acc: 0.3276 | val loss: 1.6510 | val acc: 0.4578
Epoch 02 | train loss: 1.4184 | train acc: 0.5506 | val loss: 1.3796 | val acc: 0.5203
Epoch 03 | train loss: 1.1163 | train acc: 0.6358 | val loss: 1.3249 | val acc: 0.5396
Epoch 04 | train loss: 0.9581 | train acc: 0.6892 | val loss: 1.3243 | val acc: 0.5336
Epoch 05 | train loss: 0.8525 | train acc: 0.7223 | val loss: 1.3487 | val acc: 0.5409
Epoch 06 | train loss: 0.7708 | train acc: 0.7504 | val loss: 1.3691 | val acc: 0.5349
Epoch 07 | train loss: 0.6964 | train acc: 0.7743 | val loss: 1.4090 | val acc: 0.5356
Epoch 08 | train loss: 0.6319 | train acc: 0.7971 | val loss: 1.4605 | val acc: 0.5289
Epoch 09 | train loss: 0.5806 | train acc: 0.8151 | val loss: 1.5145 | val acc: 0.5309
Epoch 10 | train loss: 0.5294 | train acc: 0.8374 | val loss: 1.5523 | val acc: 0.5303
Epoch 11 | train loss: 0.4825 | train acc: 0.8571 | val loss: 1.6064 | val acc: 0.5243
Epoch 12 | train loss: 0.4408 | train acc: 

... (2 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8406 | train acc: 0.3153 | val loss: 1.6953 | val acc: 0.4238
Epoch 02 | train loss: 1.4173 | train acc: 0.5505 | val loss: 1.3993 | val acc: 0.5190
Epoch 03 | train loss: 1.1218 | train acc: 0.6370 | val loss: 1.3302 | val acc: 0.5442
Epoch 04 | train loss: 0.9637 | train acc: 0.6790 | val loss: 1.3078 | val acc: 0.5562
Epoch 05 | train loss: 0.8542 | train acc: 0.7176 | val loss: 1.3234 | val acc: 0.5449
Epoch 06 | train loss: 0.7613 | train acc: 0.7550 | val loss: 1.3467 | val acc: 0.5469
Epoch 07 | train loss: 0.6892 | train acc: 0.7840 | val loss: 1.3805 | val acc: 0.5509
Epoch 08 | train loss: 0.6241 | train acc: 0.8044 | val loss: 1.4418 | val acc: 0.5449
Epoch 09 | train loss: 0.5687 | train acc: 0.8202 | val loss: 1.4707 | val acc: 0.5469
Epoch 10 | train loss: 0.5223 | train acc: 0.8427 | val loss: 1.5097 | val acc: 0.5489
Epoch 11 | train loss: 0.4736 | train acc: 0.8590 | val loss: 1.5712 | val acc: 0.5476
Epoch 12 | train loss: 0.4229 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8460 | train acc: 0.3007 | val loss: 1.6798 | val acc: 0.4345
Epoch 02 | train loss: 1.4354 | train acc: 0.5388 | val loss: 1.3565 | val acc: 0.5376
Epoch 03 | train loss: 1.1386 | train acc: 0.6258 | val loss: 1.2694 | val acc: 0.5549
Epoch 04 | train loss: 0.9728 | train acc: 0.6750 | val loss: 1.2461 | val acc: 0.5629
Epoch 05 | train loss: 0.8627 | train acc: 0.7159 | val loss: 1.2627 | val acc: 0.5629
Epoch 06 | train loss: 0.7710 | train acc: 0.7487 | val loss: 1.2712 | val acc: 0.5709
Epoch 07 | train loss: 0.6994 | train acc: 0.7728 | val loss: 1.3205 | val acc: 0.5642
Epoch 08 | train loss: 0.6317 | train acc: 0.7981 | val loss: 1.3258 | val acc: 0.5635
Epoch 09 | train loss: 0.5730 | train acc: 0.8227 | val loss: 1.3467 | val acc: 0.5629
Epoch 10 | train loss: 0.5180 | train acc: 0.8413 | val loss: 1.4033 | val acc: 0.5669
Epoch 11 | train loss: 0.4661 | train acc: 0.8638 | val loss: 1.4345 | val acc: 0.5655
Epoch 12 | train loss: 0.4216 | train acc: 

In [21]:
%%time
my_classifiers['mlp_tfidf-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_tfidf-enc/seed-{seed}"
    my_classifiers['mlp_tfidf-enc'].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=tfidf_encoder,
            model_option='mlp',
            save_to=run_folder,
            smlp_args=smlp_args,
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 6.322 seconds
Dataloader done! Time taken: 0.135 seconds
Training...


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9187 | train acc: 0.2487 | val loss: 1.8563 | val acc: 0.4854
Epoch 02 | train loss: 1.6908 | train acc: 0.5434 | val loss: 1.5608 | val acc: 0.5279
Epoch 03 | train loss: 1.3574 | train acc: 0.6023 | val loss: 1.3603 | val acc: 0.5445
Epoch 04 | train loss: 1.1462 | train acc: 0.6427 | val loss: 1.2788 | val acc: 0.5532
Epoch 05 | train loss: 1.0178 | train acc: 0.6710 | val loss: 1.2461 | val acc: 0.5559
Epoch 06 | train loss: 0.9336 | train acc: 0.6933 | val loss: 1.2430 | val acc: 0.5559
Epoch 07 | train loss: 0.8759 | train acc: 0.7043 | val loss: 1.2483 | val acc: 0.5499
Epoch 08 | train loss: 0.8286 | train acc: 0.7227 | val loss: 1.2624 | val acc: 0.5512
Epoch 09 | train loss: 0.7921 | train acc: 0.7350 | val loss: 1.2841 | val acc: 0.5432
Epoch 10 | train loss: 0.7556 | train acc: 0.7480 | val loss: 1.2988 | val acc: 0.5459
Epoch 11 | train loss: 0.7265 | train acc: 0.7572 | val loss: 1.3197 | val acc: 0.5485
Epoch 12 | train loss: 0.7046 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9138 | train acc: 0.3123 | val loss: 1.8483 | val acc: 0.4478
Epoch 02 | train loss: 1.6734 | train acc: 0.5466 | val loss: 1.5540 | val acc: 0.5050
Epoch 03 | train loss: 1.3420 | train acc: 0.6087 | val loss: 1.3700 | val acc: 0.5143
Epoch 04 | train loss: 1.1313 | train acc: 0.6436 | val loss: 1.2890 | val acc: 0.5343
Epoch 05 | train loss: 1.0083 | train acc: 0.6725 | val loss: 1.2694 | val acc: 0.5389
Epoch 06 | train loss: 0.9230 | train acc: 0.6997 | val loss: 1.2790 | val acc: 0.5376
Epoch 07 | train loss: 0.8639 | train acc: 0.7115 | val loss: 1.2868 | val acc: 0.5389
Epoch 08 | train loss: 0.8179 | train acc: 0.7278 | val loss: 1.3078 | val acc: 0.5389
Epoch 09 | train loss: 0.7787 | train acc: 0.7401 | val loss: 1.3376 | val acc: 0.5356
Epoch 10 | train loss: 0.7444 | train acc: 0.7507 | val loss: 1.3566 | val acc: 0.5329
Epoch 11 | train loss: 0.7198 | train acc: 0.7638 | val loss: 1.3859 | val acc: 0.5236
Epoch 12 | train loss: 0.6891 | train acc: 

... (2 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9121 | train acc: 0.3083 | val loss: 1.8497 | val acc: 0.4285
Epoch 02 | train loss: 1.6702 | train acc: 0.5172 | val loss: 1.5655 | val acc: 0.4864
Epoch 03 | train loss: 1.3436 | train acc: 0.6000 | val loss: 1.3835 | val acc: 0.5296
Epoch 04 | train loss: 1.1361 | train acc: 0.6441 | val loss: 1.2997 | val acc: 0.5476
Epoch 05 | train loss: 1.0091 | train acc: 0.6714 | val loss: 1.2731 | val acc: 0.5516
Epoch 06 | train loss: 0.9264 | train acc: 0.6960 | val loss: 1.2712 | val acc: 0.5576
Epoch 07 | train loss: 0.8677 | train acc: 0.7083 | val loss: 1.2816 | val acc: 0.5549
Epoch 08 | train loss: 0.8258 | train acc: 0.7238 | val loss: 1.2929 | val acc: 0.5469
Epoch 09 | train loss: 0.7887 | train acc: 0.7364 | val loss: 1.3122 | val acc: 0.5516
Epoch 10 | train loss: 0.7502 | train acc: 0.7500 | val loss: 1.3349 | val acc: 0.5489
Epoch 11 | train loss: 0.7249 | train acc: 0.7555 | val loss: 1.3552 | val acc: 0.5409
Epoch 12 | train loss: 0.6975 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9164 | train acc: 0.3163 | val loss: 1.8507 | val acc: 0.4824
Epoch 02 | train loss: 1.6832 | train acc: 0.5413 | val loss: 1.5452 | val acc: 0.5143
Epoch 03 | train loss: 1.3542 | train acc: 0.6010 | val loss: 1.3362 | val acc: 0.5462
Epoch 04 | train loss: 1.1490 | train acc: 0.6376 | val loss: 1.2459 | val acc: 0.5595
Epoch 05 | train loss: 1.0251 | train acc: 0.6677 | val loss: 1.2083 | val acc: 0.5755
Epoch 06 | train loss: 0.9381 | train acc: 0.6898 | val loss: 1.2010 | val acc: 0.5709
Epoch 07 | train loss: 0.8836 | train acc: 0.7021 | val loss: 1.2050 | val acc: 0.5762
Epoch 08 | train loss: 0.8333 | train acc: 0.7154 | val loss: 1.2221 | val acc: 0.5709
Epoch 09 | train loss: 0.7972 | train acc: 0.7308 | val loss: 1.2338 | val acc: 0.5689
Epoch 10 | train loss: 0.7662 | train acc: 0.7392 | val loss: 1.2575 | val acc: 0.5662
Epoch 11 | train loss: 0.7432 | train acc: 0.7469 | val loss: 1.2689 | val acc: 0.5689
Epoch 12 | train loss: 0.7164 | train acc: 

In [41]:
%%time
my_classifiers['twolayer-mlp_indicator-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/twolayer-mlp_indicator-enc/seed-{seed}"
    my_classifiers['twolayer-mlp_indicator-enc'].append(
        twolayer_cross_val_exp(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=token_indicator_encoder,
            save_to=run_folder,
            mlp_args=two_layer_mlp_args,
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Training...


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8631 | train acc: 0.2871 | val loss: 1.6790 | val acc: 0.4235
Epoch 02 | train loss: 1.3814 | train acc: 0.5344 | val loss: 1.3457 | val acc: 0.5339
Epoch 03 | train loss: 1.0326 | train acc: 0.6497 | val loss: 1.2808 | val acc: 0.5552
Epoch 04 | train loss: 0.8390 | train acc: 0.7216 | val loss: 1.3002 | val acc: 0.5565
Epoch 05 | train loss: 0.6947 | train acc: 0.7670 | val loss: 1.3763 | val acc: 0.5492
Epoch 06 | train loss: 0.5702 | train acc: 0.8172 | val loss: 1.4298 | val acc: 0.5545
Epoch 07 | train loss: 0.4664 | train acc: 0.8551 | val loss: 1.5501 | val acc: 0.5532
Epoch 08 | train loss: 0.3777 | train acc: 0.8824 | val loss: 1.6314 | val acc: 0.5465
Epoch 09 | train loss: 0.2858 | train acc: 0.9197 | val loss: 1.7463 | val acc: 0.5439
Epoch 10 | train loss: 0.2293 | train acc: 0.9388 | val loss: 1.8798 | val acc: 0.5372
Epoch 11 | train loss: 0.1801 | train acc: 0.9541 | val loss: 1.9723 | val acc: 0.5352
Epoch 12 | train loss: 0.1456 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8607 | train acc: 0.2862 | val loss: 1.6617 | val acc: 0.4291
Epoch 02 | train loss: 1.3680 | train acc: 0.5387 | val loss: 1.3611 | val acc: 0.5136
Epoch 03 | train loss: 1.0159 | train acc: 0.6582 | val loss: 1.3415 | val acc: 0.5236
Epoch 04 | train loss: 0.8359 | train acc: 0.7164 | val loss: 1.4073 | val acc: 0.5223
Epoch 05 | train loss: 0.6991 | train acc: 0.7628 | val loss: 1.4771 | val acc: 0.5196
Epoch 06 | train loss: 0.5818 | train acc: 0.8064 | val loss: 1.5563 | val acc: 0.5196
Epoch 07 | train loss: 0.4832 | train acc: 0.8425 | val loss: 1.6249 | val acc: 0.5196
Epoch 08 | train loss: 0.3935 | train acc: 0.8791 | val loss: 1.7544 | val acc: 0.5077
Epoch 09 | train loss: 0.3125 | train acc: 0.9102 | val loss: 1.8918 | val acc: 0.5116
Epoch 10 | train loss: 0.2489 | train acc: 0.9326 | val loss: 2.0214 | val acc: 0.5176
Epoch 11 | train loss: 0.1972 | train acc: 0.9488 | val loss: 2.1231 | val acc: 0.5143
Epoch 12 | train loss: 0.1511 | train acc: 

... (2 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8626 | train acc: 0.2787 | val loss: 1.6729 | val acc: 0.4271
Epoch 02 | train loss: 1.3866 | train acc: 0.5194 | val loss: 1.3485 | val acc: 0.5336
Epoch 03 | train loss: 1.0325 | train acc: 0.6453 | val loss: 1.3113 | val acc: 0.5456
Epoch 04 | train loss: 0.8447 | train acc: 0.7143 | val loss: 1.3290 | val acc: 0.5476
Epoch 05 | train loss: 0.7086 | train acc: 0.7620 | val loss: 1.3894 | val acc: 0.5436
Epoch 06 | train loss: 0.5825 | train acc: 0.8124 | val loss: 1.4572 | val acc: 0.5396
Epoch 07 | train loss: 0.4858 | train acc: 0.8472 | val loss: 1.5507 | val acc: 0.5303
Epoch 08 | train loss: 0.3852 | train acc: 0.8833 | val loss: 1.6391 | val acc: 0.5336
Epoch 09 | train loss: 0.3132 | train acc: 0.9052 | val loss: 1.7676 | val acc: 0.5269
Epoch 10 | train loss: 0.2407 | train acc: 0.9356 | val loss: 1.8734 | val acc: 0.5283
Epoch 11 | train loss: 0.1940 | train acc: 0.9506 | val loss: 1.9770 | val acc: 0.5230
Epoch 12 | train loss: 0.1443 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8662 | train acc: 0.2782 | val loss: 1.6377 | val acc: 0.4591
Epoch 02 | train loss: 1.3768 | train acc: 0.5345 | val loss: 1.3013 | val acc: 0.5323
Epoch 03 | train loss: 1.0395 | train acc: 0.6463 | val loss: 1.2529 | val acc: 0.5556
Epoch 04 | train loss: 0.8553 | train acc: 0.7055 | val loss: 1.2515 | val acc: 0.5589
Epoch 05 | train loss: 0.7215 | train acc: 0.7564 | val loss: 1.2948 | val acc: 0.5662
Epoch 06 | train loss: 0.6049 | train acc: 0.8009 | val loss: 1.3495 | val acc: 0.5582
Epoch 07 | train loss: 0.4973 | train acc: 0.8422 | val loss: 1.4543 | val acc: 0.5622
Epoch 08 | train loss: 0.4083 | train acc: 0.8776 | val loss: 1.5147 | val acc: 0.5602
Epoch 09 | train loss: 0.3191 | train acc: 0.9065 | val loss: 1.6269 | val acc: 0.5562
Epoch 10 | train loss: 0.2533 | train acc: 0.9298 | val loss: 1.7226 | val acc: 0.5542
Epoch 11 | train loss: 0.1964 | train acc: 0.9481 | val loss: 1.8383 | val acc: 0.5482
Epoch 12 | train loss: 0.1574 | train acc: 

In [42]:
%%time
my_classifiers['twolayer-mlp_bow-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/twolayer-mlp_bow-enc/seed-{seed}"
    my_classifiers['twolayer-mlp_bow-enc'].append(
        twolayer_cross_val_exp(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=bow_encoder,
            save_to=run_folder,
            mlp_args=two_layer_mlp_args,
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Training...


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8552 | train acc: 0.2836 | val loss: 1.6775 | val acc: 0.4036
Epoch 02 | train loss: 1.4240 | train acc: 0.5135 | val loss: 1.3672 | val acc: 0.5233
Epoch 03 | train loss: 1.0749 | train acc: 0.6362 | val loss: 1.3098 | val acc: 0.5532
Epoch 04 | train loss: 0.8756 | train acc: 0.7081 | val loss: 1.3278 | val acc: 0.5465
Epoch 05 | train loss: 0.7066 | train acc: 0.7675 | val loss: 1.3855 | val acc: 0.5539
Epoch 06 | train loss: 0.5761 | train acc: 0.8190 | val loss: 1.4631 | val acc: 0.5565
Epoch 07 | train loss: 0.4697 | train acc: 0.8530 | val loss: 1.5349 | val acc: 0.5519
Epoch 08 | train loss: 0.3699 | train acc: 0.8912 | val loss: 1.6581 | val acc: 0.5426
Epoch 09 | train loss: 0.2992 | train acc: 0.9160 | val loss: 1.7501 | val acc: 0.5485
Epoch 10 | train loss: 0.2294 | train acc: 0.9398 | val loss: 1.8670 | val acc: 0.5479
Epoch 11 | train loss: 0.1774 | train acc: 0.9561 | val loss: 2.0210 | val acc: 0.5406
Epoch 12 | train loss: 0.1410 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8622 | train acc: 0.2796 | val loss: 1.6939 | val acc: 0.4059
Epoch 02 | train loss: 1.4119 | train acc: 0.5189 | val loss: 1.3892 | val acc: 0.5156
Epoch 03 | train loss: 1.0660 | train acc: 0.6433 | val loss: 1.3673 | val acc: 0.5130
Epoch 04 | train loss: 0.8677 | train acc: 0.7043 | val loss: 1.3785 | val acc: 0.5349
Epoch 05 | train loss: 0.7254 | train acc: 0.7575 | val loss: 1.4400 | val acc: 0.5309
Epoch 06 | train loss: 0.6068 | train acc: 0.7989 | val loss: 1.5444 | val acc: 0.5243
Epoch 07 | train loss: 0.5026 | train acc: 0.8403 | val loss: 1.6646 | val acc: 0.5203
Epoch 08 | train loss: 0.4029 | train acc: 0.8761 | val loss: 1.7411 | val acc: 0.5196
Epoch 09 | train loss: 0.3189 | train acc: 0.9047 | val loss: 1.8787 | val acc: 0.5143
Epoch 10 | train loss: 0.2516 | train acc: 0.9305 | val loss: 1.9616 | val acc: 0.5063
Epoch 11 | train loss: 0.2076 | train acc: 0.9430 | val loss: 2.1269 | val acc: 0.5136
Epoch 12 | train loss: 0.1664 | train acc: 

... (2 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8705 | train acc: 0.2576 | val loss: 1.7043 | val acc: 0.4032
Epoch 02 | train loss: 1.4313 | train acc: 0.5032 | val loss: 1.4027 | val acc: 0.5110
Epoch 03 | train loss: 1.0884 | train acc: 0.6316 | val loss: 1.3313 | val acc: 0.5403
Epoch 04 | train loss: 0.8750 | train acc: 0.7013 | val loss: 1.3631 | val acc: 0.5336
Epoch 05 | train loss: 0.7257 | train acc: 0.7532 | val loss: 1.4046 | val acc: 0.5283
Epoch 06 | train loss: 0.5802 | train acc: 0.8111 | val loss: 1.4798 | val acc: 0.5236
Epoch 07 | train loss: 0.4737 | train acc: 0.8563 | val loss: 1.5586 | val acc: 0.5383
Epoch 08 | train loss: 0.3721 | train acc: 0.8892 | val loss: 1.6991 | val acc: 0.5269
Epoch 09 | train loss: 0.2968 | train acc: 0.9173 | val loss: 1.8199 | val acc: 0.5150
Epoch 10 | train loss: 0.2230 | train acc: 0.9413 | val loss: 1.9484 | val acc: 0.5210
Epoch 11 | train loss: 0.1743 | train acc: 0.9569 | val loss: 2.0544 | val acc: 0.5190
Epoch 12 | train loss: 0.1356 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8608 | train acc: 0.2824 | val loss: 1.6720 | val acc: 0.4145
Epoch 02 | train loss: 1.4125 | train acc: 0.5119 | val loss: 1.3519 | val acc: 0.5223
Epoch 03 | train loss: 1.0754 | train acc: 0.6370 | val loss: 1.2761 | val acc: 0.5469
Epoch 04 | train loss: 0.8733 | train acc: 0.7025 | val loss: 1.2808 | val acc: 0.5635
Epoch 05 | train loss: 0.7284 | train acc: 0.7567 | val loss: 1.3211 | val acc: 0.5702
Epoch 06 | train loss: 0.6007 | train acc: 0.8059 | val loss: 1.3872 | val acc: 0.5689
Epoch 07 | train loss: 0.4883 | train acc: 0.8462 | val loss: 1.4628 | val acc: 0.5655
Epoch 08 | train loss: 0.3903 | train acc: 0.8809 | val loss: 1.5888 | val acc: 0.5649
Epoch 09 | train loss: 0.3129 | train acc: 0.9087 | val loss: 1.6640 | val acc: 0.5569
Epoch 10 | train loss: 0.2451 | train acc: 0.9335 | val loss: 1.7868 | val acc: 0.5595
Epoch 11 | train loss: 0.1935 | train acc: 0.9513 | val loss: 1.8996 | val acc: 0.5542
Epoch 12 | train loss: 0.1523 | train acc: 

In [43]:
%%time
my_classifiers['twolayer-mlp_tfidf-enc'] = []
for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/twolayer-mlp_tfidf-enc/seed-{seed}"
    my_classifiers['twolayer-mlp_tfidf-enc'].append(
        twolayer_cross_val_exp(
            data_splits=my_cross_validation_sets[run],
            outcome_space=agnews_outcome_space,
            encoder=tfidf_encoder,
            save_to=run_folder,
            mlp_args=two_layer_mlp_args,
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Training...


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9125 | train acc: 0.2696 | val loss: 1.7987 | val acc: 0.3677
Epoch 02 | train loss: 1.5365 | train acc: 0.4830 | val loss: 1.3856 | val acc: 0.5013
Epoch 03 | train loss: 1.1755 | train acc: 0.6018 | val loss: 1.2829 | val acc: 0.5432
Epoch 04 | train loss: 0.9986 | train acc: 0.6549 | val loss: 1.2781 | val acc: 0.5459
Epoch 05 | train loss: 0.8937 | train acc: 0.6913 | val loss: 1.2809 | val acc: 0.5499
Epoch 06 | train loss: 0.8110 | train acc: 0.7207 | val loss: 1.3129 | val acc: 0.5499
Epoch 07 | train loss: 0.7451 | train acc: 0.7430 | val loss: 1.3548 | val acc: 0.5432
Epoch 08 | train loss: 0.6844 | train acc: 0.7713 | val loss: 1.4110 | val acc: 0.5472
Epoch 09 | train loss: 0.6310 | train acc: 0.7869 | val loss: 1.4502 | val acc: 0.5505
Epoch 10 | train loss: 0.5752 | train acc: 0.8076 | val loss: 1.5044 | val acc: 0.5452
Epoch 11 | train loss: 0.5283 | train acc: 0.8272 | val loss: 1.5508 | val acc: 0.5492
Epoch 12 | train loss: 0.4824 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9209 | train acc: 0.2395 | val loss: 1.8311 | val acc: 0.4251
Epoch 02 | train loss: 1.5353 | train acc: 0.4936 | val loss: 1.3979 | val acc: 0.5070
Epoch 03 | train loss: 1.1504 | train acc: 0.6057 | val loss: 1.3130 | val acc: 0.5230
Epoch 04 | train loss: 0.9779 | train acc: 0.6661 | val loss: 1.3308 | val acc: 0.5176
Epoch 05 | train loss: 0.8586 | train acc: 0.7075 | val loss: 1.3457 | val acc: 0.5303
Epoch 06 | train loss: 0.7783 | train acc: 0.7392 | val loss: 1.3841 | val acc: 0.5336
Epoch 07 | train loss: 0.7143 | train acc: 0.7587 | val loss: 1.4327 | val acc: 0.5269
Epoch 08 | train loss: 0.6593 | train acc: 0.7760 | val loss: 1.4954 | val acc: 0.5230
Epoch 09 | train loss: 0.5957 | train acc: 0.8001 | val loss: 1.5518 | val acc: 0.5130
Epoch 10 | train loss: 0.5482 | train acc: 0.8176 | val loss: 1.6218 | val acc: 0.5143
Epoch 11 | train loss: 0.4972 | train acc: 0.8364 | val loss: 1.6715 | val acc: 0.5143
Epoch 12 | train loss: 0.4431 | train acc: 

... (2 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9156 | train acc: 0.2445 | val loss: 1.8239 | val acc: 0.3965
Epoch 02 | train loss: 1.5231 | train acc: 0.5029 | val loss: 1.4011 | val acc: 0.5196
Epoch 03 | train loss: 1.1461 | train acc: 0.6070 | val loss: 1.3151 | val acc: 0.5429
Epoch 04 | train loss: 0.9727 | train acc: 0.6636 | val loss: 1.2976 | val acc: 0.5509
Epoch 05 | train loss: 0.8576 | train acc: 0.7071 | val loss: 1.3286 | val acc: 0.5476
Epoch 06 | train loss: 0.7886 | train acc: 0.7279 | val loss: 1.3641 | val acc: 0.5429
Epoch 07 | train loss: 0.7211 | train acc: 0.7540 | val loss: 1.4264 | val acc: 0.5396
Epoch 08 | train loss: 0.6598 | train acc: 0.7781 | val loss: 1.4618 | val acc: 0.5422
Epoch 09 | train loss: 0.6101 | train acc: 0.7979 | val loss: 1.5245 | val acc: 0.5336
Epoch 10 | train loss: 0.5615 | train acc: 0.8124 | val loss: 1.5824 | val acc: 0.5256
Epoch 11 | train loss: 0.5152 | train acc: 0.8299 | val loss: 1.6421 | val acc: 0.5303
Epoch 12 | train loss: 0.4640 | train acc: 

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9215 | train acc: 0.2576 | val loss: 1.8378 | val acc: 0.4178
Epoch 02 | train loss: 1.5461 | train acc: 0.4938 | val loss: 1.3439 | val acc: 0.5223
Epoch 03 | train loss: 1.1610 | train acc: 0.5985 | val loss: 1.2322 | val acc: 0.5589
Epoch 04 | train loss: 0.9936 | train acc: 0.6594 | val loss: 1.2153 | val acc: 0.5722
Epoch 05 | train loss: 0.8880 | train acc: 0.6905 | val loss: 1.2301 | val acc: 0.5762
Epoch 06 | train loss: 0.8011 | train acc: 0.7243 | val loss: 1.2707 | val acc: 0.5682
Epoch 07 | train loss: 0.7309 | train acc: 0.7500 | val loss: 1.2955 | val acc: 0.5695
Epoch 08 | train loss: 0.6702 | train acc: 0.7732 | val loss: 1.3392 | val acc: 0.5642
Epoch 09 | train loss: 0.6169 | train acc: 0.7903 | val loss: 1.3901 | val acc: 0.5635
Epoch 10 | train loss: 0.5591 | train acc: 0.8176 | val loss: 1.4508 | val acc: 0.5576
Epoch 11 | train loss: 0.5076 | train acc: 0.8327 | val loss: 1.4891 | val acc: 0.5576
Epoch 12 | train loss: 0.4566 | train acc: 

In [ ]:
part1.analysis.plot_box_and_points(part1.analysis.make_data_frame(my_classifiers))

# Bayesian estimation and ROPE

In [ ]:
# First, a little helper, to gather all accuracies across runs and folds
def get_all_measurements_for(data, sys_name):
    return np.array([list(run_data['accuracies'].values()) for run_data in data[sys_name]])

# this will have shape [num_runs, num_folds]
get_all_measurements_for(my_classifiers, 'logistic_indicator-enc')

In [ ]:


ROPE = 0.005 

def compare_all_pairs(classifiers, rope_size=ROPE, runs=1, plot=False):
    rows = []
    for name_a, name_b in combinations(classifiers.keys(), 2):
        posterior = part1.analysis.BenavoliCorrelatedT(
            get_all_measurements_for(classifiers, name_a).flatten(),
            get_all_measurements_for(classifiers, name_b).flatten(),
            runs=runs,
            names=[name_a, name_b],
        )
        p_a, p_tie, p_b = posterior.probs(rope_size)
        rows.append({
            "A": name_a,
            "B": name_b,
            "mean diff (B-A)": posterior.mean_diff,
            "P(A wins)": p_a,
            "P(tie / in ROPE)": p_tie,
            "P(B wins)": p_b,
            "effect size": posterior.effect_size,
        })
        if plot:
            posterior.plot(rope_size)
    return pd.DataFrame(rows)

results_df = compare_all_pairs(my_classifiers, rope_size=ROPE, runs=len(my_seeds))
results_df.style.format({
    "mean diff (B-A)": "{:+.4f}",
    "P(A wins)": "{:.3f}",
    "P(tie / in ROPE)": "{:.3f}",
    "P(B wins)": "{:.3f}",
    "effect size": "{:.2f}",
})


In [ ]:
posterior = part1.analysis.BenavoliCorrelatedT(
    get_all_measurements_for(my_classifiers, 'logistic_bow-enc').flatten(),
    get_all_measurements_for(my_classifiers, 'mlp_bow-enc').flatten(),
    runs=len(my_seeds),
    names=['logistic_bow-enc', 'mlp_bow-enc'],
)
_ = posterior.plot(ROPE)
